# 6 Generating text

In [24]:
import torch

def top_p_filter(probs, p):                                                                
    """Nucleus filtering (Eq. 24): keep the smallest set of top tokens whose total prob >= p.
                                                                                           
    probs: (vocab_size,) probabilities that sum to 1                                       
    returns: (vocab_size,) same shape, zeros outside the kept set, renormalized to sum to 1
                                                                                           
    Example: probs = [0.15, 0.5, 0.05, 0.3], p = 0.7                                       
        sorted high -> low: 0.5, 0.3, 0.15, 0.05                                           
        0.5 alone < 0.7, 0.5 + 0.3 = 0.8 >= 0.7  -> keep {0.5, 0.3}                        
        result: [0, 0.625, 0, 0.375]   (0.5 / 0.8, 0.3 / 0.8, in their ORIGINAL positions)
    """                                                                                    
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)                   
    cum_probs = torch.cumsum(sorted_probs, dim=-1)                                      
    idx = (cum_probs < p).sum() + 1                                                     
    filter_probs = torch.zeros_like(probs)                                              
    filter_probs[sorted_indices[:idx]] = sorted_probs[:idx]                             
    return filter_probs / filter_probs.sum(dim=-1)                                                 
probs = torch.tensor([0.15, 0.5, 0.05, 0.3])
p = 0.7
top_p_filter(probs, p)

tensor([0.0000, 0.6250, 0.0000, 0.3750])

# 4.2 SGD Optimizer

In [16]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or 0.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.
        return loss

In [17]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1)
for t in range(100):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.

23.83068084716797
22.886985778808594
22.244220733642578
21.733478546142578
21.300983428955078
20.921646118164062
20.581388473510742
20.271404266357422
19.985736846923828
19.72014808654785
19.471494674682617
19.237369537353516
19.0158748626709
18.805500030517578
18.604995727539062
18.413341522216797
18.22966766357422
18.05324363708496
17.883438110351562
17.719703674316406
17.561567306518555
17.408613204956055
17.260469436645508
17.116806030273438
16.977333068847656
16.841787338256836
16.70992660522461
16.581541061401367
16.45643424987793
16.334426879882812
16.215354919433594
16.09906768798828
15.985431671142578
15.874319076538086
15.765607833862305
15.65919303894043
15.554970741271973
15.452851295471191
15.352741241455078
15.254563331604004
15.158238410949707
15.063693046569824
14.970861434936523
14.879678726196289
14.79008674621582
14.7020263671875
14.615447044372559
14.530296325683594
14.446527481079102
14.364092826843262
14.282951354980469
14.20306396484375
14.124388694763184
14.0468

# 4.1 Cross-entropy loss

In [12]:
from cs336_basics.training import cross_entropy
import torch
import torch.nn.functional as F

# 2 positions, vocab of 3
logits = torch.tensor([[1.0, 2.0, 3.0],
                       [3.0, 2.0, 1.0]])  # (2, 3) = (batch, vocab)
targets = torch.tensor([2, 0])            # (2,)   both point at the largest logit

# steps inside cross_entropy
z = logits - logits.max(dim=-1, keepdim=True).values            # (2, 3)
print('z:\n', z)
logsum = torch.log(torch.exp(z).sum(-1, keepdim=True))
print('logsum:\n', logsum)
log_probs = z - logsum   # (2, 3)
print('log_probs:\n', log_probs)
print('log_probs | targets:\n', log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1))  # (2,) the two target log-probs

# verify with pytorch
print(cross_entropy(logits, targets))
print(F.cross_entropy(logits, targets))

z:
 tensor([[-2., -1.,  0.],
        [ 0., -1., -2.]])
logsum:
 tensor([[0.4076],
        [0.4076]])
log_probs:
 tensor([[-2.4076, -1.4076, -0.4076],
        [-0.4076, -1.4076, -2.4076]])
log_probs | targets:
 tensor([-0.4076, -0.4076])
tensor(0.4076)
tensor(0.4076)


# 3.5 The Full Transformer LM

## Resource accounting

In [21]:
def resource_accounting(
    vocab_size, context_length, num_layers, d_model, num_heads, d_ff
):
    """Rule: Given A in R^{m x n} and B in R^{n x p}, the matrix product AB requires 2mnp FLOPs.

    PyTorch Convention:
      - Input activations are row tensors: X has shape (batch_size, context_length, in_features).
      - Linear layers store weights as (out_features, in_features) via nn.Linear(in_features, out_features).
      - Forward pass computes X @ weight.T, performing:
          (context_length, in_features) @ (in_features, out_features) -> (context_length, out_features).

    Dimensions & Parameter Shapes:
      - X:                          (context_length, d_model)
      - W_q, W_k, W_v, W_o.weight:  (d_model, d_model)
      - W1.weight (gate_proj):      (d_ff, d_model)
      - W3.weight (up_proj):        (d_ff, d_model)
      - W2.weight (down_proj):      (d_model, d_ff)
      - lm_head.weight:             (vocab_size, d_model)

    Intermediate Runtime Activations:
      - Q, K, V = X @ W_{q,k,v}.T:  (context_length, d_model) -> reshaped to (num_heads, context_length, d_k)
      - Q_h @ K_h.T:                (context_length, d_k) @ (d_k, context_length) -> (context_length, context_length)
      - Attn_h @ V_h:               (context_length, context_length) @ (context_length, d_k) -> (context_length, d_k)
      - Attn_out @ W_o.T:           (context_length, d_model) @ (d_model, d_model) -> (context_length, d_model)
      - X @ W1.T, X @ W3.T:         (context_length, d_model) @ (d_model, d_ff) -> (context_length, d_ff)
      - FFN_hidden @ W2.T:          (context_length, d_ff) @ (d_ff, d_model) -> (context_length, d_model)
      - X @ lm_head.T:              (context_length, d_model) @ (d_model, vocab_size) -> (context_length, vocab_size)
    """
    # Attention projections: Q, K, V
    qkv_projx = 3 * 2 * (context_length * d_model**2)

    # Attention core
    # Q @ K^T across all heads: sum_h 2 * T * d_k * T = 2 * T^2 * d_model
    atten1 = 2 * (context_length**2 * d_model)
    # (Q @ K^T) @ V across all heads: sum_h 2 * T * T * d_k = 2 * T^2 * d_model
    atten2 = 2 * (context_length**2 * d_model)
    # Output projection: Attn_out @ W_o.T -> 2 * T * d_model * d_model
    atten3 = 2 * (context_length * d_model**2)
    atten = num_layers * (atten1 + atten2 + atten3 + qkv_projx)
    print(f"Attention: {atten/1e12:.3f} TFLOPs")

    # Feed-forward networks using SwiGLU: W2(SiLU(X @ W1.T) * (X @ W3.T))
    # X @ W3.T
    ffn1 = 2 * context_length * d_model * d_ff
    # X @ W1.T
    ffn2 = 2 * context_length * d_model * d_ff
    # intermediate @ W2.T
    ffn3 = 2 * context_length * d_ff * d_model
    ffn = num_layers * (ffn1 + ffn2 + ffn3)
    print(f"FFN: {ffn/1e12:.3f} TFLOPs")

    # LM head: X @ lm_head.weight.T
    lm_head = 2 * vocab_size * d_model * context_length
    print(f"LM head: {lm_head/1e12:.3f} TFLOPs")

    total = atten + ffn + lm_head
    print(f"Total: {total/1e12:.3f} TFLOPs")

    return total

In [22]:
resource_accounting(
    vocab_size=50257, 
    context_length=1024, 
    num_layers=48, 
    d_model=1600, 
    num_heads=25, 
    d_ff=4288)

Attention: 1.329 TFLOPs
FFN: 2.023 TFLOPs
LM head: 0.165 TFLOPs
Total: 3.517 TFLOPs


3516769894400

## 3.4.4 Scaled Dot-Product Attention

In [11]:
from cs336_basics.model import softmax
import torch

x = torch.Tensor([[1.0, 2.0, 3.0], 
                  [2.0, 2.0, 3.0], 
                  [3.0, 2.0, 3.0]])
print(softmax(x, 0))  # dim=0: for each column, normalize across rows → each column sums to 1
print(softmax(x, 1))  # dim=1: for each row, normalize across columns → each row sums to 1

tensor([[0.0900, 0.3333, 0.3333],
        [0.2447, 0.3333, 0.3333],
        [0.6652, 0.3333, 0.3333]])
tensor([[0.0900, 0.2447, 0.6652],
        [0.2119, 0.2119, 0.5761],
        [0.4223, 0.1554, 0.4223]])


## 3.4.3 Relative Positional Ebemddings

In [28]:
from cs336_basics.model import RotaryPositionalEmbedding
theta = 10.0
d_k = 4
max_seq_len = 8
rope = RotaryPositionalEmbedding(theta, d_k, max_seq_len)
print(rope.cos_cache)
print(rope.sin_cache)

tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.5403,  0.9504,  0.9504],
        [-0.4161, -0.4161,  0.8066,  0.8066],
        [-0.9900, -0.9900,  0.5828,  0.5828],
        [-0.6536, -0.6536,  0.3011,  0.3011],
        [ 0.2837,  0.2837, -0.0103, -0.0103],
        [ 0.9602,  0.9602, -0.3208, -0.3208],
        [ 0.7539,  0.7539, -0.5994, -0.5994]])
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.8415,  0.3110,  0.3110],
        [ 0.9093,  0.9093,  0.5911,  0.5911],
        [ 0.1411,  0.1411,  0.8126,  0.8126],
        [-0.7568, -0.7568,  0.9536,  0.9536],
        [-0.9589, -0.9589,  0.9999,  0.9999],
        [-0.2794, -0.2794,  0.9471,  0.9471],
        [ 0.6570,  0.6570,  0.8004,  0.8004]])


In [27]:
print(rope.cos_cache.shape)
token_positions = torch.tensor(
    [[0, 1, 2],
     [1, 2, 3],
     [2, 3, 4]],
    dtype=torch.long
)
print(token_positions.shape)
cos = rope.cos_cache[token_positions]
print(cos)
print(cos.shape)

torch.Size([8, 4])
torch.Size([3, 3])
tensor([[[ 1.0000,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.5403,  0.9504,  0.9504],
         [-0.4161, -0.4161,  0.8066,  0.8066]],

        [[ 0.5403,  0.5403,  0.9504,  0.9504],
         [-0.4161, -0.4161,  0.8066,  0.8066],
         [-0.9900, -0.9900,  0.5828,  0.5828]],

        [[-0.4161, -0.4161,  0.8066,  0.8066],
         [-0.9900, -0.9900,  0.5828,  0.5828],
         [-0.6536, -0.6536,  0.3011,  0.3011]]])
torch.Size([3, 3, 4])


In [32]:
# (..., seq_len, d_k)
x = torch.randn(3, 3, 4, dtype=torch.float32)
result = rope(x, token_positions)
print(result.shape)
print(result)

torch.Size([3, 3, 4])
tensor([[[-0.8158,  0.7119,  0.2699,  0.5510],
         [ 0.7164,  0.9044, -0.2547, -1.6661],
         [-0.9847,  0.6290, -1.4866,  1.2716]],

        [[-0.1304, -2.3228,  0.7871, -3.0229],
         [-0.3627, -1.0440,  1.5032, -0.0546],
         [-0.9078, -0.8257,  0.1198,  0.9635]],

        [[-0.4825,  0.1576,  1.3566,  2.4191],
         [-0.0854,  1.4661, -0.2463, -1.0697],
         [ 1.4251,  1.1910, -0.8716, -0.3678]]])


# 3.2 Einops

In [3]:
import torch
from einops import rearrange, einsum
print('Torch:', torch.__version__); 
print('MPS Available:', torch.backends.mps.is_available())

Torch: 2.13.0
MPS Available: True


## einstein_example2

In [10]:
import torch
from einops import rearrange, einsum

images = torch.randn(64, 128, 128, 3)
dim_by = torch.linspace(start=0.0, end=1.0, steps=10)

# Method 1: Reshape and multiply
dim_value = rearrange(dim_by, "dim_value -> 1 dim_value 1 1 1")
images_rearr = rearrange(images, "b height width channel -> b 1 height width channel")
dimmed_images_1 = images_rearr * dim_value

# Method 2: einsum in one go
dimmed_images_2 = einsum(
    images, dim_by,
    "batch height width channel, dim_value -> batch dim_value height width channel"
)

# Assertion
assert torch.allclose(dimmed_images_1, dimmed_images_2), "Outputs do not match!"

## einstein_example3

In [16]:
channels_last = torch.randn(64, 32, 32, 3) # (batch, height, width, channel)
B = torch.randn(32*32, 32*32) # linear tranformation
print(channels_last.size(0))
print(channels_last.size(1))
print(channels_last.size(2))
print(channels_last.size(3))

64
32
32
3


In [20]:
# traditional way
## Rearrange an image tensor for mixing across all pixels
channels_last_flat = channels_last.view(
-1, channels_last.size(1) * channels_last.size(2), channels_last.size(3)
)
channels_first_flat = channels_last_flat.transpose(1, 2)
channels_first_flat_transformed = channels_first_flat @ B.T
channels_last_flat_transformed = channels_first_flat_transformed.transpose(1, 2)
channels_last_transformed = channels_last_flat_transformed.view(*channels_last.shape)

In [19]:
# einops way
height = width = 32
channels_first = rearrange(
    channels_last,
    "batch height width channel -> batch channel (height width)"
)
channels_first_transformed = einsum(
    channels_first, B,
    "batch channel pixel_in, pixel_out pixel_in -> batch channel pixel_out"
)
channels_last_transformed = rearrange(
    channels_first_transformed,
    "batch channel (height width) -> batch height width channel",
    height=height, width=width
)

# 2.5 Experimenting with BPE Tokenizer Training

In [14]:
# code from assignment
with open('data/TinyStoriesV2-GPT4-valid.txt', "rb") as f:
    num_processes = 4
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        # Run pre-tokenization on your chunk and store the counts for each pre-token

## Parallel Pretokenization

In [1]:
from cs336_basics.pretokenization import parallel_pretokenize

# multiprocessing
num_processes = 4
filepath = 'data/TinyStoriesV2-GPT4-valid.txt'
special_tokens = ["<|endoftext|>"]
word_freq = parallel_pretokenize(filepath, special_tokens, num_processes)
word_freq

{(b'u',): 1,
 (b' ', b'd', b'o', b'n'): 2771,
 (b"'", b't'): 11552,
 (b' ', b'h', b'a', b'v', b'e'): 10504,
 (b' ', b't', b'o'): 150493,
 (b' ', b'b', b'e'): 12777,
 (b' ', b's', b'c', b'a', b'r', b'e', b'd'): 6370,
 (b' ', b'o', b'f'): 25467,
 (b' ', b't', b'h', b'e'): 211031,
 (b' ', b'l', b'o', b'u', b'd'): 2152,
 (b' ', b'd', b'o', b'g'): 13043,
 (b',',): 235432,
 (b' ', b'I'): 18481,
 (b"'", b'l', b'l'): 454,
 (b' ', b'p', b'r', b'o', b't', b'e', b'c', b't'): 246,
 (b' ', b'y', b'o', b'u'): 28401,
 (b'"', b'.'): 799,
 (b' ', b'T', b'h', b'e'): 46977,
 (b' ', b'm', b'o', b'l', b'e'): 166,
 (b' ', b'f', b'e', b'l', b't'): 8707,
 (b' ', b's', b'o'): 18387,
 (b' ', b's', b'a', b'f', b'e'): 2591,
 (b' ', b'w', b'i', b't', b'h'): 42981,
 (b' ', b'l', b'i', b't', b't', b'l', b'e'): 23934,
 (b' ', b'g', b'i', b'r', b'l'): 12422,
 (b'.',): 421616,
 (b' ', b'S', b'h', b'e'): 38040,
 (b' ', b'w', b'a', b's'): 108019,
 (b' ', b'v', b'e', b'r', b'y'): 25453,
 (b' ', b'k', b'i', b'n', b'd'): 30

# 2.4 BPE Tokenizer Training

## Pre-tokenization from GPT-2

In [257]:
import regex as re
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
re.findall(PAT, "some text that I'll pre-tokenize")

['some', ' text', ' that', ' I', "'ll", ' pre', '-', 'tokenize']

## improved bpe_example

In [5]:
from pprint import pprint
def get_tuple_bytes(word):
    """
    convert a word into a tuple of its individual UTF-8 bytes
    e.g. "low" -> (b'l', b'o', b'w')
    @Input: word: str
    @Return: tuple[bytes, ...]
    """
    word_bytes = word.encode("utf-8")
    return tuple(bytes([b]) for b in word_bytes)
    
def get_freq(text):
    """
    build a frequency table
    @Input: text: str
    @Return: dict[tuple[bytes, ...], int]
    """
    freq = {}
    for word in text.split(" "):
        tuple_bytes = get_tuple_bytes(word)
        # inquiry the key, if nothing, default to 0
        freq[tuple_bytes] = freq.get(tuple_bytes, 0) + 1
    return freq

def get_pair_index(freq):
    """
    index the pair of bytes
    @Input: freq: dict[tuple[bytes, ...], int]
    @Return: pair_freq: dict[tuple[bytes, bytes], int]
             words: list[list[bytes]]
             counts: list[int]
             pair_index: dict[tuple[bytes, bytes], set[int]]
    """
    pair_freq = {}
    words = []
    counts = []
    pair_index = {}
    for i, (tuple_bytes,num) in enumerate(freq.items()):
        words.append(list(tuple_bytes))
        counts.append(num)
        for pair in zip(tuple_bytes, tuple_bytes[1:]):
            pair_freq[pair] = pair_freq.get(pair, 0) + num
            if pair not in pair_index:
                pair_index[pair] = set()
            pair_index[pair].add(i)
    return pair_freq, words, counts, pair_index

def merge_pair(word, max_pair):                                                            
    """                                                                                    
    merging (b'l', b'o') in [b'l', b'o', b'w'] -> [b'lo', b'w']                            
    @Input: word: list[bytes]                                                              
            max_pair: tuple[bytes, bytes]                                                  
    @Return: new_word: list[bytes]                                                         
    """                                                                                                                                                                
    new_word = []                                                                          
    i = 0                                                                                  
    while i < len(word):                                                                   
        if (i + 1) < len(word) and (word[i], word[i + 1]) == max_pair:                     
            new_word.append(word[i] + word[i + 1])                                         
            i = i + 2                                                                      
        else:                                                                              
            new_word.append(word[i])                                                       
            i = i + 1                                                                      
    return new_word 

def merge(max_pair, pair_freq, words, counts, pair_index):
    """
    merge the max_pair and update pair_freq, words, counts, pair_index
    e.g. merging (b'l', b'o') in (b'l', b'o', b'w') -> (b'lo', b'w')
    @Input: max_pair: tuple[bytes, bytes]
            pair_freq: dict[tuple[bytes, bytes], int]
            words: list[list[bytes]]
            counts: list[int]
            pair_index: dict[tuple[bytes, bytes], set[int]]
    @Return: new_token: bytes
    """
    new_token = max_pair[0] + max_pair[1]
    for index in list(pair_index[max_pair]):
        word = words[index]
        count = counts[index]
        new_word = merge_pair(word, max_pair)
        # update words
        words[index] = new_word
        # update the pair_freq and pair_index
        old_pairs = {}
        for pair in zip(word, word[1:]):
            old_pairs[pair] = old_pairs.get(pair, 0) + 1
        new_pairs = {}
        for pair in zip(new_word, new_word[1:]):
            new_pairs[pair] = new_pairs.get(pair, 0) + 1
        # remove old pairs from old word
        for pair, num in old_pairs.items():
            pair_freq[pair] -= num * count
            if pair_freq[pair] <= 0:
                del pair_freq[pair]
            pair_index[pair].discard(index)
            if not pair_index[pair]:
                del pair_index[pair]
        # add new pairs from new word
        for pair, num in new_pairs.items():
            pair_freq[pair] = pair_freq.get(pair, 0) + num * count
            if pair not in pair_index:
                pair_index[pair] = set()
            pair_index[pair].add(index)
    return new_token 

In [6]:
text = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"
freq = get_freq(text)
pair_freq, words, counts, pair_index = get_pair_index(freq)
print(words)
print(counts)
pprint(pair_freq)
pprint(pair_index)

[[b'l', b'o', b'w'], [b'l', b'o', b'w', b'e', b'r'], [b'w', b'i', b'd', b'e', b's', b't'], [b'n', b'e', b'w', b'e', b's', b't']]
[5, 2, 3, 6]
{(b'd', b'e'): 3,
 (b'e', b'r'): 2,
 (b'e', b's'): 9,
 (b'e', b'w'): 6,
 (b'i', b'd'): 3,
 (b'l', b'o'): 7,
 (b'n', b'e'): 6,
 (b'o', b'w'): 7,
 (b's', b't'): 9,
 (b'w', b'e'): 8,
 (b'w', b'i'): 3}
{(b'd', b'e'): {2},
 (b'e', b'r'): {1},
 (b'e', b's'): {2, 3},
 (b'e', b'w'): {3},
 (b'i', b'd'): {2},
 (b'l', b'o'): {0, 1},
 (b'n', b'e'): {3},
 (b'o', b'w'): {0, 1},
 (b's', b't'): {2, 3},
 (b'w', b'e'): {1, 3},
 (b'w', b'i'): {2}}


In [7]:
# first merge
max_pair = max(pair_freq, key=lambda p: (pair_freq[p], p))
print("pair to merge: ", max_pair)
new_token = merge(max_pair, pair_freq, words, counts, pair_index)
print(words)
print(counts)
pprint(pair_freq)
pprint(pair_index)

pair to merge:  (b's', b't')
[[b'l', b'o', b'w'], [b'l', b'o', b'w', b'e', b'r'], [b'w', b'i', b'd', b'e', b'st'], [b'n', b'e', b'w', b'e', b'st']]
[5, 2, 3, 6]
{(b'd', b'e'): 3,
 (b'e', b'r'): 2,
 (b'e', b'st'): 9,
 (b'e', b'w'): 6,
 (b'i', b'd'): 3,
 (b'l', b'o'): 7,
 (b'n', b'e'): 6,
 (b'o', b'w'): 7,
 (b'w', b'e'): 8,
 (b'w', b'i'): 3}
{(b'd', b'e'): {2},
 (b'e', b'r'): {1},
 (b'e', b'st'): {2, 3},
 (b'e', b'w'): {3},
 (b'i', b'd'): {2},
 (b'l', b'o'): {0, 1},
 (b'n', b'e'): {3},
 (b'o', b'w'): {0, 1},
 (b'w', b'e'): {1, 3},
 (b'w', b'i'): {2}}


## bpe_example

In [31]:
def get_tuple_bytes(word):
    """
    convert a word into a tuple of its individual UTF-8 bytes
    e.g. "low" -> (b'l', b'o', b'w')
    @Input: word: str
    @Return: tuple[bytes, ...]
    """
    word_bytes = word.encode("utf-8")
    return tuple(bytes([b]) for b in word_bytes)
    
def get_freq(text):
    """
    build a frequency table
    @Input: text: str
    @Return: dict[tuple[bytes, ...], int]
    """
    freq = {}
    for word in text.split(" "):
        tuple_bytes = get_tuple_bytes(word)
        # inquiry the key, if nothing, default to 0
        freq[tuple_bytes] = freq.get(tuple_bytes, 0) + 1
    return freq

def get_successive(freq):
    """
    get the successive pair and count the frequency
    find the pair with highest frequency
    @Input: freq: dict[tuple[bytes, ...], int]
    @Return: tuple[dict[tuple[bytes, bytes], int], tuple[bytes, bytes]]
    """
    pair_freq = {}
    for tuple_bytes,num in freq.items():
        for pair in zip(tuple_bytes, tuple_bytes[1:]):
            pair_freq[pair] = pair_freq.get(pair, 0) + num
    # sort it by highest frequency and in lexicographically
    max_pair = max(pair_freq, key=lambda p: (pair_freq[p], p))
    return pair_freq, max_pair

def merge(freq, pair):
    """
    merge the frequency table with the pair
    e.g. merging (b'l', b'o') in (b'l', b'o', b'w') -> (b'lo', b'w')
    @Input: freq: dict[tuple[bytes, ...], int]
            pair: tuple[bytes, bytes]
    @Return: dict[tuple[bytes, ...], int]
    """
    merged_freq = {}
    for tuple_bytes,num in freq.items():
        new_tuple = []
        i = 0
        while i < len(tuple_bytes):
            if (i+1) < len(tuple_bytes) and (tuple_bytes[i], tuple_bytes[i+1]) == pair:
                new_tuple.append(tuple_bytes[i] + tuple_bytes[i+1])
                i = i + 2
            else:
                new_tuple.append(tuple_bytes[i])
                i = i + 1
        merged_freq[tuple(new_tuple)] = num
    return merged_freq

def merge_pair(word, max_pair):                                                            
    """                                                                                    
    merging (b'l', b'o') in [b'l', b'o', b'w'] -> [b'lo', b'w']                            
    @Input: word: list[bytes]                                                              
            max_pair: tuple[bytes, bytes]                                                  
    @Return: new_word: list[bytes]                                                         
    """                                                                                                                                                                
    new_word = []                                                                          
    i = 0                                                                                  
    while i < len(word):                                                                   
        if (i + 1) < len(word) and (word[i], word[i + 1]) == max_pair:                     
            new_word.append(word[i] + word[i + 1])                                         
            i = i + 2                                                                      
        else:                                                                              
            new_word.append(word[i])                                                       
            i = i + 1                                                                      
    return new_word 

def tokenize(word, vocabulary):
    """
    tokenize a word using a learned (ordered) list of BPE merges
    @Input: word: str
            vocabulary: list[tuple[bytes, bytes]]  (merges, in learned order)
    @Return: tuple[bytes, ...]
    """
    word_bytes = [bytes([b]) for b in word.encode("utf-8")]

    for pair in vocabulary:
        if len(word_bytes) < 2:
            break
        new_word = []
        i = 0
        while i < len(word_bytes):
            if (i + 1) < len(word_bytes) and (word_bytes[i], word_bytes[i + 1]) == pair:
                new_word.append(word_bytes[i] + word_bytes[i + 1])
                i = i + 2
            else:
                new_word.append(word_bytes[i])
                i = i + 1
        word_bytes = new_word

    return new_word

### In a loop

In [32]:
text = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"
freq = get_freq(text)
vocabulary = []
i = 0
while i < 6:
    pair_freq, max_pair = get_successive(freq)
    vocabulary.append(max_pair)
    # merge
    print('Merge the pair of', max_pair)
    freq = merge(freq, max_pair)
    i = i + 1
print(vocabulary)
print(tokenize('newest', vocabulary))

Merge the pair of (b's', b't')
Merge the pair of (b'e', b'st')
Merge the pair of (b'o', b'w')
Merge the pair of (b'l', b'ow')
Merge the pair of (b'w', b'est')
Merge the pair of (b'n', b'e')
[(b's', b't'), (b'e', b'st'), (b'o', b'w'), (b'l', b'ow'), (b'w', b'est'), (b'n', b'e')]
[b'ne', b'west']


### Sample text

In [9]:
text = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"
freq = get_freq(text)
print(freq)
vocabulary = []

{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b's', b't'): 3, (b'n', b'e', b'w', b'e', b's', b't'): 6}


### First merge

In [10]:
# get the successive pair
pair_freq, max_pair = get_successive(freq)
print(pair_freq)
vocabulary.append(max_pair)

# merge
print('Merge the pair of', max_pair)
freq = merge(freq, max_pair)
print(freq)

{(b'l', b'o'): 7, (b'o', b'w'): 7, (b'w', b'e'): 8, (b'e', b'r'): 2, (b'w', b'i'): 3, (b'i', b'd'): 3, (b'd', b'e'): 3, (b'e', b's'): 9, (b's', b't'): 9, (b'n', b'e'): 6, (b'e', b'w'): 6}
Merge the pair of (b's', b't')
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'e', b'st'): 3, (b'n', b'e', b'w', b'e', b'st'): 6}


### Second merge

In [11]:
# get the succesive pair
pair_freq, max_pair = get_successive(freq)
print(pair_freq)
vocabulary.append(max_pair)

# merge
print('Merge the pair of', max_pair)
freq = merge(freq, max_pair)
print(freq)

{(b'l', b'o'): 7, (b'o', b'w'): 7, (b'w', b'e'): 8, (b'e', b'r'): 2, (b'w', b'i'): 3, (b'i', b'd'): 3, (b'd', b'e'): 3, (b'e', b'st'): 9, (b'n', b'e'): 6, (b'e', b'w'): 6}
Merge the pair of (b'e', b'st')
{(b'l', b'o', b'w'): 5, (b'l', b'o', b'w', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}


### Third merge

In [12]:
# get the succesive pair
pair_freq, max_pair = get_successive(freq)
print(pair_freq)
vocabulary.append(max_pair)

# merge
print('Merge the pair of', max_pair)
freq = merge(freq, max_pair)
print(freq)

{(b'l', b'o'): 7, (b'o', b'w'): 7, (b'w', b'e'): 2, (b'e', b'r'): 2, (b'w', b'i'): 3, (b'i', b'd'): 3, (b'd', b'est'): 3, (b'n', b'e'): 6, (b'e', b'w'): 6, (b'w', b'est'): 6}
Merge the pair of (b'o', b'w')
{(b'l', b'ow'): 5, (b'l', b'ow', b'e', b'r'): 2, (b'w', b'i', b'd', b'est'): 3, (b'n', b'e', b'w', b'est'): 6}


In [13]:
print(vocabulary)

[(b's', b't'), (b'e', b'st'), (b'o', b'w')]


# 2.1 The Unicode Standard

In [1]:
chr(0)

'\x00'

In [2]:
print(chr(0))

 


In [3]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [4]:
print("this is a test" + chr(0) + "string")

this is a test string


# 2.2 Unicode Encodings

In [8]:
test_string = "hello! 方"
utf8_encoded = test_string.encode("utf-8") # str -> bytes
print(utf8_encoded) # hexadecimal
print(type(utf8_encoded))
# one byte != one character
print(list(utf8_encoded))
print(utf8_encoded.decode("utf-8")) # bytes -> str
print(len(test_string))
print(len(utf8_encoded))

b'hello! \xe6\x96\xb9'
<class 'bytes'>
[104, 101, 108, 108, 111, 33, 32, 230, 150, 185]
hello! 方
8
10


In [14]:
test_string = "牛逼"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded) # hexadecimal
print(list(utf8_encoded))
print(b'\xe7\x89\x9b'.decode("utf-8"))

b'\xe7\x89\x9b\xe9\x80\xbc'
[231, 137, 155, 233, 128, 188]
牛


In [14]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [15]:
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [16]:
decode_utf8_bytes_to_str_wrong("方".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe6 in position 0: unexpected end of data

In [11]:
test_string = "hello! 方"
utf16_encoded = test_string.encode("utf-16")
print(utf16_encoded)
print(type(utf16_encoded))
print(utf8_encoded.decode("utf-16"))

b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00\xb9e'
<class 'bytes'>
hello! 方


In [12]:
list(utf16_encoded)

[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 185, 101]

In [13]:
print(len(test_string))
print(len(utf16_encoded))

8
18
